[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/S10_02_mlp_keras.ipynb)

# S10 · Práctica 2 — Tu primer MLP en Keras: cancelaciones y precio por noche

**Tiempo estimado:** 1 h 30 min · **Sprint 10** (UD 4.2)

**Contexto TurisData.** Con `reservas_turisdata.csv` ya has predicho cancelaciones con modelos
clásicos. Ahora usarás un **perceptrón multicapa (MLP)** de Keras para dos tareas:
1. **Clasificación:** ¿se cancelará la reserva? (`cancelada`).
2. **Regresión:** ¿cuál es el `precio_noche`?

**Al terminar sabrás:**
1. Preparar datos para una red (escalar números, codificar categorías, separar entrenamiento/validación/prueba).
2. Construir, compilar y entrenar un MLP con Keras (`Sequential`, `compile`, `fit`).
3. Elegir la capa de salida, la pérdida y las métricas según sea clasificación o regresión.
4. Leer la curva de aprendizaje y evaluar el modelo en datos de prueba frente a una referencia sencilla.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # menos mensajes de TensorFlow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression, Ridge

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)   # fija NumPy, Python y TensorFlow a la vez
print("TensorFlow", tf.__version__)

## 1. Preparar los datos para una red neuronal

**Idea.** Las redes son sensibles a la escala: si `antelacion_dias` va de 0 a 400 y `adultos` de 1 a 4,
la primera "grita" más que la segunda. Por eso:
- las variables **numéricas** se **estandarizan** (media 0, desviación 1);
- las **categóricas** se convierten en columnas 0/1 (*one-hot*);
- el escalador se **ajusta solo con el entrenamiento** (si no, "espiaríamos" los datos de prueba: fuga de datos).

Además crearemos `mes_llegada` a partir de la fecha (la temporada influye) y **no** usaremos
`id_reserva` ni las fechas en bruto.

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
print(df.shape, "· tasa de cancelación:", round(df["cancelada"].mean(), 3))
df.head(3)

**Ejercicio 1.** Define dos listas: `cols_num` (las columnas numéricas: `antelacion_dias`, `noches`, `adultos`, `ninos`,
`peticiones_especiales`, `cliente_repetidor`, `cancelaciones_previas`, `mes_llegada`) y `cols_cat`
(las categóricas: `tipo_habitacion`, `regimen`, `canal`, `pais_origen`, `tarifa`, `deposito`).
Para la clasificación añadiremos `precio_noche` a las numéricas más abajo.

In [ ]:
# TODO: Dos listas de nombres de columna, sin id_reserva, sin fechas y sin el objetivo
...

assert len(cols_num) == 8 and len(cols_cat) == 6, "Deben ser 8 numéricas y 6 categóricas"
assert all(c in df.columns for c in cols_num + cols_cat), "Alguna columna no existe: revisa los nombres"
assert "cancelada" not in cols_num + cols_cat and "id_reserva" not in cols_num, "No incluyas el objetivo ni el identificador"

### Separar en entrenamiento, validación y prueba
- **Entrenamiento (64 %)**: donde la red ajusta sus pesos.
- **Validación (16 %)**: para vigilar durante el entrenamiento y tomar decisiones (cuántas épocas, qué arquitectura).
- **Prueba (20 %)**: se toca **una sola vez**, al final, para dar la cifra honesta.

In [ ]:
def preparar(objetivo, extra_num=()):
    """Devuelve matrices ya escaladas/codificadas y el preprocesador ajustado."""
    numericas = list(cols_num) + list(extra_num)
    X = df[numericas + cols_cat]
    y = df[objetivo].to_numpy()
    estrato = y if objetivo == "cancelada" else None
    X_res, X_pru, y_res, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=estrato)
    estrato = y_res if objetivo == "cancelada" else None
    X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=SEMILLA, stratify=estrato)
    pre = ColumnTransformer([("num", StandardScaler(), numericas),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat)])
    A_ent = pre.fit_transform(X_ent).astype("float32")     # ajusta SOLO con entrenamiento
    A_val = pre.transform(X_val).astype("float32")
    A_pru = pre.transform(X_pru).astype("float32")
    return A_ent, A_val, A_pru, y_ent, y_val, y_pru, pre

# Clasificación: el precio es un dato conocido al reservar, así que puede ser una variable de entrada
Xc_ent, Xc_val, Xc_pru, yc_ent, yc_val, yc_pru, pre_c = preparar("cancelada", extra_num=["precio_noche"])
print("Clasificación:", Xc_ent.shape, Xc_val.shape, Xc_pru.shape)

## 2. Un MLP para clasificar cancelaciones

**Idea.** Un MLP es una pila de capas `Dense` (todas las neuronas de una capa conectadas con todas las de la siguiente).
Para clasificación binaria:
- **Capas ocultas:** `relu`.
- **Capa de salida:** 1 neurona con `sigmoid` (probabilidad de cancelar).
- **Pérdida:** `binary_crossentropy` (la misma que programaste a mano).
- **Métricas:** `AUC` y `accuracy`. Recuerda que la referencia de "no saber nada" es acertar ≈ 54 % (no cancela).

**Ejercicio 2.** Escribe `construir_mlp_clasificacion(n_entradas, capas=(32, 16))` que devuelva un modelo
`keras.Sequential` con una capa `Input`, una capa `Dense` con `relu` por cada número de `capas`, y una
salida `Dense(1, activation="sigmoid")`. Debe **compilarlo** con el optimizador `adam`, la pérdida
`binary_crossentropy` y las métricas `AUC` (llamada `auc`) y `accuracy`.

In [ ]:
# TODO: Recorre `capas` añadiendo Dense(unidades, activation="relu"); termina con Dense(1, "sigmoid")
...

mlp_c = construir_mlp_clasificacion(Xc_ent.shape[1])
mlp_c.summary()
assert len(mlp_c.layers) == 3, "Con capas=(32, 16) debe haber 3 capas Dense (2 ocultas + salida)"
assert mlp_c.layers[-1].activation.__name__ == "sigmoid", "La salida debe ser sigmoid"
assert mlp_c.count_params() == Xc_ent.shape[1] * 32 + 32 + 32 * 16 + 16 + 16 + 1, "Número de parámetros inesperado: revisa capas y tamaños"

### Entrenar (`fit`)
- **época**: una pasada completa por los datos de entrenamiento;
- **lote (*batch*)**: cuántas filas se usan antes de actualizar los pesos (aquí 64);
- `validation_data`: datos para vigilar en cada época **sin entrenar con ellos**.

In [ ]:
hist_c = mlp_c.fit(Xc_ent, yc_ent, validation_data=(Xc_val, yc_val), epochs=30, batch_size=64, verbose=0)

def dibujar_curvas(hist, metrica, titulo):
    h = hist.history
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
    ejes[0].plot(h["loss"], label="entrenamiento"); ejes[0].plot(h["val_loss"], label="validación")
    ejes[0].set_title("Pérdida"); ejes[0].set_xlabel("época"); ejes[0].set_ylabel("pérdida"); ejes[0].legend()
    ejes[1].plot(h[metrica], label="entrenamiento"); ejes[1].plot(h["val_" + metrica], label="validación")
    ejes[1].set_title(metrica.upper()); ejes[1].set_xlabel("época"); ejes[1].set_ylabel(metrica); ejes[1].legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

dibujar_curvas(hist_c, "auc", "MLP de cancelaciones: curvas de aprendizaje")

**Lee las curvas.** La pérdida de entrenamiento sigue bajando, pero la de validación se estanca y luego sube:
la red empieza a **memorizar** el entrenamiento. Es sobreajuste. En la siguiente práctica lo controlaremos.

### Evaluar en prueba y comparar con una referencia
Una cifra sola no dice nada: hay que compararla con un modelo sencillo. Usaremos una **regresión logística**.

In [ ]:
prob_mlp = mlp_c.predict(Xc_pru, verbose=0).ravel()
logistica = LogisticRegression(max_iter=1000).fit(Xc_ent, yc_ent)
prob_log = logistica.predict_proba(Xc_pru)[:, 1]

tabla = pd.DataFrame({
    "modelo": ["Siempre 'no cancela'", "Regresión logística", "MLP (32-16)"],
    "AUC": [0.5, roc_auc_score(yc_pru, prob_log), roc_auc_score(yc_pru, prob_mlp)],
    "precisión": [1 - yc_pru.mean(), accuracy_score(yc_pru, prob_log > 0.5), accuracy_score(yc_pru, prob_mlp > 0.5)],
}).round(3)
tabla

**Ejercicio 3.** Guarda en `auc_mlp` el AUC del MLP en prueba y en `auc_log` el de la regresión logística
(a partir de `prob_mlp` y `prob_log`, con `roc_auc_score`). Después guarda en `mlp_gana` un booleano
(`True`/`False`) que diga si el MLP supera a la regresión logística **por más de 0.02 de AUC**.
Esta última respuesta puede ser `False`: no siempre gana la red.

In [ ]:
# TODO: roc_auc_score(yc_pru, probabilidades) y una comparación con margen 0.02
...

assert 0.6 < auc_mlp < 0.9 and 0.6 < auc_log < 0.9, "Los AUC deberían estar entre 0.6 y 0.9. ¿Has usado las probabilidades y no las clases?"
assert isinstance(mlp_gana, bool), "mlp_gana debe ser True o False"
print(f"AUC MLP = {auc_mlp:.3f} · AUC logística = {auc_log:.3f} · ¿gana la red por más de 0.02? {mlp_gana}")

## 3. Un MLP para predecir el precio por noche (regresión)

**Idea.** Cambian tres cosas respecto a la clasificación:
- **Salida:** 1 neurona **sin activación** (puede valer cualquier número).
- **Pérdida:** `mse` (error cuadrático medio); la métrica legible es `mae` (error medio absoluto, en euros).
- **Objetivo escalado:** entrenamos con el precio estandarizado y luego **deshacemos** la transformación
  para dar el error en euros.

Ojo: aquí `precio_noche` es el objetivo, así que **no** puede estar entre las variables de entrada.

In [ ]:
Xr_ent, Xr_val, Xr_pru, yr_ent, yr_val, yr_pru, pre_r = preparar("precio_noche")
media_p, desv_p = yr_ent.mean(), yr_ent.std()
esc = lambda v: ((v - media_p) / desv_p).astype("float32")     # a escala estandarizada
desesc = lambda v: v * desv_p + media_p                         # de vuelta a euros

**Ejercicio 4.** Escribe `construir_mlp_regresion(n_entradas, capas=(32, 16))`: igual que el de clasificación pero con
salida `Dense(1)` **sin activación**, pérdida `"mse"` y métrica `"mae"`.

In [ ]:
# TODO: Misma estructura, pero la última capa no lleva activation y compila con loss="mse", metrics=["mae"]
...

mlp_r = construir_mlp_regresion(Xr_ent.shape[1])
assert mlp_r.layers[-1].activation.__name__ == "linear", "La salida de regresión no lleva activación"
assert mlp_r.loss == "mse", "Compila con loss='mse'"

hist_r = mlp_r.fit(Xr_ent, esc(yr_ent), validation_data=(Xr_val, esc(yr_val)), epochs=30, batch_size=64, verbose=0)
dibujar_curvas(hist_r, "mae", "MLP de precio_noche: curvas de aprendizaje (escala estandarizada)")

In [ ]:
pred_mlp = desesc(mlp_r.predict(Xr_pru, verbose=0).ravel())
ridge = Ridge().fit(Xr_ent, yr_ent)
pred_ridge = ridge.predict(Xr_pru)
mae = lambda p: mean_absolute_error(yr_pru, p)
tabla_r = pd.DataFrame({
    "modelo": ["Siempre el precio medio", "Regresión lineal (Ridge)", "MLP (32-16)"],
    "MAE (€)": [mae(np.full(len(yr_pru), yr_ent.mean())), mae(pred_ridge), mae(pred_mlp)],
}).round(2)
tabla_r

**Ejercicio 5.** Calcula en `mae_mlp_euros` el error medio absoluto **en euros** del MLP sobre el conjunto de prueba
(usa `pred_mlp`, que ya está en euros, e `yr_pru`). Calcula también `mejora_vs_media`: en qué fracción
(0 a 1) reduce el error respecto a predecir siempre el precio medio del entrenamiento.

In [ ]:
# TODO: mean_absolute_error(yr_pru, pred_mlp) y 1 - mae_mlp/mae_base
...

assert 5 < mae_mlp_euros < 20, "El MAE debería ser de unos 12 €. ¿Has comparado con yr_pru en euros?"
assert 0.3 < mejora_vs_media < 0.8, "El MLP debería reducir el error a menos de la mitad de la referencia"

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(yr_pru, pred_mlp, s=8, alpha=0.4)
lim = [yr_pru.min(), yr_pru.max()]
plt.plot(lim, lim, "r--", label="predicción perfecta")
plt.xlabel("precio real (€/noche)"); plt.ylabel("precio predicho (€/noche)")
plt.title("MLP de precio: real frente a predicho"); plt.legend(); plt.show()

## 4. Mini-reto integrador

Construye un MLP de clasificación **más ancho** (`capas=(128, 64)`), entrénalo 15 épocas con `batch_size=64`
y guarda en `auc_ancho` su AUC en **validación** (la última época: `hist_ancho.history["val_auc"][-1]`).
¿Es mejor la red grande? Escribe tu conclusión en la celda de texto.
Pista de la teoría: más parámetros = más capacidad, pero también más facilidad para memorizar.

In [ ]:
# TODO: construir_mlp_clasificacion(n, capas=(128, 64)).fit(..., epochs=15, validation_data=...)
...

assert mlp_ancho.count_params() > mlp_c.count_params() * 3, "La red ancha debe tener muchos más parámetros"
assert len(hist_ancho.history["val_auc"]) == 15, "Entrena 15 épocas"
print(f"AUC en validación de la red ancha: {auc_ancho:.3f} · parámetros: {mlp_ancho.count_params()} frente a {mlp_c.count_params()}")

**Tu conclusión (edita esta celda):** …

## 5. Preguntas de reflexión
1. ¿Por qué se ajusta el `StandardScaler` solo con los datos de entrenamiento?
2. ¿Qué activación y qué pérdida usarías para predecir *tres* categorías de huésped? ¿Y para predecir el número de noches?
3. En la clasificación, la regresión logística rinde parecido al MLP. ¿Qué ventajas de la logística
   pesarías a favor de TurisData (coste, explicabilidad, mantenimiento)?
4. ¿Por qué el error de la regresión se da en euros y no en la escala estandarizada?

## 6. Resumen: qué has aprendido
- Redes y datos tabulares: escalar numéricas, *one-hot* para categóricas, escalador ajustado solo con entrenamiento.
- MLP en Keras: `Sequential` → `Dense` → `compile` → `fit` con validación.
- Clasificación: salida sigmoide + entropía cruzada + AUC. Regresión: salida lineal + MSE/MAE.
- Una curva de validación que se estanca o sube mientras la de entrenamiento baja es señal de sobreajuste.
- Toda red debe compararse con una referencia sencilla antes de celebrarla.